# 06 · Chunked Prefill 与尾延迟

前两章解决的是**吞吐**，这一章解决**延迟**——而且是延迟里最要命的那个指标：**尾延迟（p99）**。

场景很常见：服务正在稳定处理一批 decode 请求，突然来了一条 4K token 的长 prompt。

## 本章会推翻一个常见误解

很多人以为 chunked prefill 是 vLLM 的一个独立特性。不是。看完这章你会看到：

> **它只是第 04 章那个 `Scheduler` 的 `max_num_batched_tokens` 参数调小了而已。**

不需要新代码路径，不需要新数据结构。长 prompt 之所以被切成多轮，是因为每轮的 token 预算装不下它——`num_computed_tokens` 这个字段会自动记录进度，下一轮接着算。

这就是第 04 章强调"读懂 `num_computed_tokens` 就抓住了主干"的原因。

In [ ]:
# ===== 引导单元：环境检查 + 测量工具 + MiniGPT（每章自带，直接运行）=====
# 说明：本单元在每个 notebook 里都有一份完整副本，目的是让任何一个 notebook
#       都能在 Colab 里零配置独立运行。想改模型结构，请改 tools/build_notebooks.py
#       里的 SETUP_CODE，然后重跑编译脚本。
#
# 架构对齐：下面这套推理核心刻意模仿了 vLLM V1 的模块划分与命名，
#   详见 docs/vllm-mapping.md 的对照表。
#       EngineCore.step()           ←→ vllm/v1/engine/core.py
#         ├─ Scheduler.schedule()   ←→ vllm/v1/core/sched/scheduler.py
#         ├─ ModelRunner.execute_model() ←→ vllm/v1/worker/gpu_model_runner.py
#         └─ Scheduler.update_from_output()
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MiniGPT 只有 2700 万参数，用 float16 跑在 GPU 上；CPU 上 float16 很慢，用 float32
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32


def sync():
    """GPU 是异步执行的，计时前必须同步，否则测到的是下发时间不是执行时间。"""
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def bench(fn, warmup=3, iters=10):
    """返回单次调用的平均耗时（毫秒）。warmup 用来排除首次 kernel 编译等开销。"""
    for _ in range(warmup):
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    sync()
    return (time.perf_counter() - t0) / iters * 1000.0


def peak_mem_mb():
    """当前 CUDA 峰值显存占用（MB）。"""
    if DEVICE != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated() / 1024 ** 2


def reset_peak():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()


class Config:
    def __init__(self, vocab_size=50257, block_size=1024, n_layer=4, n_head=6, n_embd=384):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head


class CausalSelfAttention(nn.Module):
    """因果自注意力，支持 KV cache。

    past_kv 传入历史的 (k, v)，本步只为新 token 计算 Q/K/V，然后拼在历史后面。
    返回 (输出, 更新后的 (k, v))，其中 k/v 的 shape 是 (B, n_head, 总长度, head_dim)。
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.head_dim
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x, past_kv=None, attn_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)

        S = k.size(2)  # 总长度 = 历史 + 本步新增
        if attn_mask is None:
            # 默认因果掩码：本步第 i 个 query 的绝对位置是 S-T+i，只能看见 <= 它的 key
            mask = torch.ones(T, S, device=x.device).tril(diagonal=S - T).bool()
        else:
            # 外部传入的掩码，用于一个 batch 里混合不同进度的序列（第 04、06 章）
            mask = attn_mask
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), (k, v)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, past_kv=None, attn_mask=None):
        h, present = self.attn(self.ln_1(x), past_kv, attn_mask)
        x = x + h
        x = x + self.mlp(self.ln_2(x))
        return x, present


class MiniGPT(nn.Module):
    """极简 GPT，结构与 Llama 同源：pre-norm + 因果注意力 + 4 倍扩张 MLP + 权重共享。

    与 Llama 的两处差异：
      - 用可学习位置编码代替 RoPE（简化实现，不影响调度实验的结论）
      - 没有 GQA（本仓库是 MHA，第 03 章会手工比较两者的 KV cache 大小）
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # 权重共享，省一份 embedding 参数

        def init(m):
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

        self.apply(init)

    def forward(self, idx, past_kvs=None, pos_offset=0, attn_mask=None):
        """idx: (B, T) 的 token id。

        past_kvs: 长度等于层数的列表，每项是 (k, v)；None 表示从零开始（prefill）。
        pos_offset: 本次输入的第一个 token 的绝对位置。传 int 表示整个 batch 用同一个
                    偏移；传 shape (B,) 的张量表示每条序列各用各的偏移——当 batch 里
                    混合了不同进度的请求时必须这样传。
        attn_mask: 可选的自定义注意力掩码，用于屏蔽填充位。
        """
        B, T = idx.shape
        if torch.is_tensor(pos_offset):
            pos = pos_offset.view(B, 1) + torch.arange(T, device=idx.device)[None, :]
        else:
            pos = torch.arange(pos_offset, pos_offset + T, device=idx.device)[None, :].expand(B, T)
        x = self.wte(idx) + self.wpe(pos)

        presents = []
        for i, blk in enumerate(self.blocks):
            past = None if past_kvs is None else past_kvs[i]
            x, present = blk(x, past, attn_mask)
            presents.append(present)
        return self.lm_head(self.ln_f(x)), presents

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def build_model(seed=0, device=DEVICE, dtype=DTYPE, **kw):
    torch.manual_seed(seed)
    cfg = Config(**kw)
    model = MiniGPT(cfg).to(device=device, dtype=dtype)
    return model.eval()


@torch.no_grad()
def generate_naive(model, idx, max_new_tokens):
    """不用 KV cache：每一步都把完整序列重新算一遍（O(n^2) 重算）。"""
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.cfg.block_size:])
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens):
    """用 KV cache：prompt 只 prefill 一次，之后每步只喂 1 个 token。"""
    logits, past = model(idx)
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    out = [nxt]
    pos = idx.size(1)
    for _ in range(max_new_tokens - 1):
        logits, past = model(nxt, past_kvs=past, pos_offset=pos)
        pos += 1
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out.append(nxt)
    return torch.cat([idx] + out, dim=1)


def kv_bytes(n_layer, n_kv_head, head_dim, seq_len, batch=1, dtype_bytes=2):
    """KV cache 字节数。注意是 2（K 和 V 各一份）。"""
    return 2 * n_layer * n_kv_head * head_dim * seq_len * batch * dtype_bytes


# ========== 以下是模仿 vLLM V1 架构的推理核心 ==========


class Request:
    """对应 vllm/v1/request.py 的 Request。

    num_computed_tokens 是 vLLM 里最核心的一个字段：它记录这条请求已经有
    多少 token 的 KV 被算过。prefill、chunked prefill、前缀缓存命中——
    三种看起来完全不同的场景，在 vLLM 里都只是「把 num_computed_tokens 往前推」。
    理解这一点，chunked prefill 就不再是独立机制，而是这个字段的自然结果。
    """

    def __init__(self, request_id, prompt_token_ids, max_tokens):
        self.request_id = request_id
        self.prompt_token_ids = list(prompt_token_ids)
        self.max_tokens = max_tokens
        self.output_token_ids = []
        self.num_computed_tokens = 0
        self.status = "waiting"      # waiting / running / finished
        # 本仓库简化：直接把 KV 张量挂在请求上。
        # 真实 vLLM 不这么做——请求只持有 block_table，物理 block 由 KVCacheManager 管（第 05 章）。
        self.past = None

    @property
    def num_prompt_tokens(self):
        return len(self.prompt_token_ids)

    def all_token_ids(self):
        return self.prompt_token_ids + self.output_token_ids

    def num_tokens_to_schedule(self):
        """还欠多少 token 没算：prefill 阶段是剩余 prompt 长度，decode 阶段是 1。"""
        if self.num_computed_tokens < self.num_prompt_tokens:
            return self.num_prompt_tokens - self.num_computed_tokens
        return 1

    @property
    def is_finished(self):
        return len(self.output_token_ids) >= self.max_tokens

    def __repr__(self):
        return (f"Request({self.request_id}, computed={self.num_computed_tokens}"
                f"/{self.num_prompt_tokens}, out={len(self.output_token_ids)}"
                f"/{self.max_tokens}, {self.status})")


class SchedulerOutput:
    """对应 vllm/v1/core/sched/output.py 的 SchedulerOutput。

    调度与执行之间唯一的接口。真实 vLLM 里这个结构还包含 block 分配结果、
    抢占列表等字段，这里只保留最必要的两个。
    """

    def __init__(self, scheduled_reqs, num_scheduled_tokens):
        self.scheduled_reqs = scheduled_reqs
        self.num_scheduled_tokens = num_scheduled_tokens   # {request_id: n}

    def __len__(self):
        return len(self.scheduled_reqs)


class Scheduler:
    """对应 vllm/v1/core/sched/scheduler.py 的 Scheduler。

    职责边界是这个架构里最值得学的一点：Scheduler 只决定
    「这一轮跑哪些请求、各自跑几个 token」，它既不碰显存也不碰模型。

        显存分配 → KVCacheManager（第 05 章）
        真正计算 → ModelRunner

    三个模块分离，才能各自独立替换实现。面试被问「说说 vLLM 的架构」时，
    先把这个职责划分讲清楚，比背模块名有用得多。
    """

    def __init__(self, max_num_seqs=8, max_num_batched_tokens=2048):
        self.waiting = []
        self.running = []
        self.finished = []
        self.max_num_seqs = max_num_seqs
        # 这个预算就是 chunked prefill 的开关：调小它，长 prompt 自然被切成多轮（第 06 章）
        self.max_num_batched_tokens = max_num_batched_tokens
        self.step_id = 0

    def add_request(self, req):
        self.waiting.append(req)

    def has_unfinished(self):
        return bool(self.waiting or self.running)

    def schedule(self):
        scheduled, num_tokens = [], {}
        budget = self.max_num_batched_tokens

        # 第一优先：正在跑的请求。已进 decode 的排 1 个 token；
        # 还在做 chunked prefill 的按剩余量排，但受 budget 限制。
        for req in list(self.running):
            if budget <= 0 or len(scheduled) >= self.max_num_seqs:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n

        # 第二优先：从队列里补新请求进来做 prefill
        for req in list(self.waiting):
            if budget <= 0 or len(scheduled) >= self.max_num_seqs:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n
            self.waiting.remove(req)
            req.status = "running"
            self.running.append(req)

        self.step_id += 1
        return SchedulerOutput(scheduled, num_tokens)

    def update_from_output(self, sched_out, sampled):
        """对应 vLLM 的 update_from_output：写回采样结果，处理完成与回收。

        本轮被调度但没产生 token 的请求（比如 chunked prefill 的中间块）
        不会出现在 sampled 里，它们保持 running，下一轮继续。
        """
        for req in sched_out.scheduled_reqs:
            if req.request_id not in sampled:
                continue
            req.output_token_ids.append(sampled[req.request_id])
            if req.is_finished:
                req.status = "finished"
                if req in self.running:
                    self.running.remove(req)
                self.finished.append(req)
                req.past = None      # 简化回收；真实 vLLM 走 KVCacheManager.free()


class ModelRunner:
    """对应 vllm/v1/worker/gpu_model_runner.py 的 GPUModelRunner。

    职责：把 Scheduler 排好的一批请求拼成一次前向，返回新采样的 token。

    与真实 vLLM 的差距（要如实知道）：
      · vLLM 用 block_table 让每条序列的 KV 物理上不连续，所以不需要填充；
        这里用「右填充 + 逐序列掩码」对齐，会浪费显存——第 05 章解决。
      · vLLM 会把 prefill 和 decode 混在同一个 batch 里跑；这里分成两组处理，
        纯粹是为了让代码可读，结论不受影响。
      · 输入准备、CUDA graph、attention metadata 这些都被省掉了。
    """

    def __init__(self, model):
        self.model = model

    @torch.no_grad()
    def _run_decode_batch(self, reqs):
        """把一批进度不同的 decode 请求拼成一次前向。"""
        B = len(reqs)
        lens = [r.num_computed_tokens for r in reqs]
        Lmax = max(lens)
        n_layer = self.model.cfg.n_layer

        padded = []
        for layer in range(n_layer):
            ks, vs = [], []
            for r in reqs:
                k, v = r.past[layer]
                pad = Lmax - k.size(2)
                if pad:
                    k = F.pad(k, (0, 0, 0, pad))
                    v = F.pad(v, (0, 0, 0, pad))
                ks.append(k)
                vs.append(v)
            padded.append((torch.cat(ks, 0), torch.cat(vs, 0)))

        # 逐序列掩码：真实历史 [0, L_i) + 新 token 落在下标 Lmax
        S = Lmax + 1
        mask = torch.zeros(B, 1, 1, S, dtype=torch.bool, device=DEVICE)
        for i, r in enumerate(reqs):
            mask[i, 0, 0, : lens[i]] = True
            mask[i, 0, 0, Lmax] = True

        ids = torch.tensor([[r.all_token_ids()[r.num_computed_tokens]] for r in reqs],
                           device=DEVICE)
        pos = torch.tensor(lens, device=DEVICE)
        logits, past = self.model(ids, past_kvs=padded, pos_offset=pos, attn_mask=mask)

        sampled = {}
        for i, r in enumerate(reqs):
            rebuilt = []
            for layer in range(n_layer):
                k_all, v_all = past[layer]
                k = torch.cat([k_all[i:i + 1, :, : lens[i]],
                               k_all[i:i + 1, :, Lmax:Lmax + 1]], dim=2)
                v = torch.cat([v_all[i:i + 1, :, : lens[i]],
                               v_all[i:i + 1, :, Lmax:Lmax + 1]], dim=2)
                rebuilt.append((k, v))
            r.past = rebuilt
            r.num_computed_tokens += 1
            sampled[r.request_id] = int(logits[i, -1].argmax(-1).item())
        return sampled

    @torch.no_grad()
    def execute_model(self, sched_out):
        decode_reqs, prefill_reqs = [], []
        for r in sched_out.scheduled_reqs:
            # 判断依据是「prompt 算完了没有」，而不是「本轮排了几个 token」
            if r.num_computed_tokens >= r.num_prompt_tokens:
                decode_reqs.append(r)
            else:
                prefill_reqs.append(r)

        sampled = {}
        if decode_reqs:
            sampled.update(self._run_decode_batch(decode_reqs))

        for r in prefill_reqs:
            n = sched_out.num_scheduled_tokens[r.request_id]
            start = r.num_computed_tokens
            chunk = r.all_token_ids()[start:start + n]
            toks = torch.tensor([chunk], device=DEVICE)
            logits, past = self.model(toks, past_kvs=r.past, pos_offset=start)
            r.past = past
            r.num_computed_tokens += len(chunk)
            # 只有 prompt 全部算完，才能采样第一个输出 token
            if r.num_computed_tokens >= r.num_prompt_tokens:
                sampled[r.request_id] = int(logits[:, -1].argmax(-1).item())
        return sampled


class EngineCore:
    """对应 vllm/v1/engine/core.py 的 EngineCore。

    整个 vLLM 的推理服务就跑在这三步上：

        schedule()            决定这一轮跑什么
        execute_model()       跑模型
        update_from_output()  把结果写回请求状态

    读懂这个循环你就抓住了 vLLM 的主干。后面所有优化——chunked prefill、
    前缀缓存、抢占、投机解码——都是在这三步里插桩。
    """

    def __init__(self, model, scheduler=None):
        self.scheduler = scheduler or Scheduler()
        self.runner = ModelRunner(model)
        self.step_id = 0
        self.steps = 0

    def step(self):
        sched_out = self.scheduler.schedule()
        if len(sched_out) == 0:
            return None
        sampled = self.runner.execute_model(sched_out)
        self.scheduler.update_from_output(sched_out, sampled)
        self.step_id += 1
        self.steps += 1
        return sampled

    def run(self, max_steps=10000):
        while self.scheduler.has_unfinished() and self.steps < max_steps:
            self.step()
        return self.steps


print(f"引导单元加载完成 | device={DEVICE} dtype={DTYPE} torch={torch.__version__}")
# ===== 引导单元结束 =====

In [ ]:
model = build_model(block_size=4096)

## 一、先看问题的严重程度

长 prefill 是一次无法中断的大矩阵乘法。在它执行期间 GPU 被它独占，其他请求只能等着。这就是**队头阻塞（head-of-line blocking）**。

In [ ]:
dec_ids = torch.randint(0, model.cfg.vocab_size, (8, 128), device=DEVICE)
_, dec_past = model(dec_ids)
dec_next = torch.randint(0, model.cfg.vocab_size, (8, 1), device=DEVICE)
dec_ms = bench(lambda: model(dec_next, past_kvs=dec_past, pos_offset=128), warmup=3, iters=10)

print(f"单个 decode step（batch=8）: {dec_ms:.1f} ms\n")
for L in [256, 1024, 2048]:
    long_ids = torch.randint(0, model.cfg.vocab_size, (1, L), device=DEVICE)
    ms = bench(lambda: model(long_ids), warmup=2, iters=5)
    print(f"prefill {L:>5} token : {ms:>8.1f} ms   （是单个 decode step 的 {ms / dec_ms:>5.1f} 倍）")

print()
print("一条 2048 token 的 prefill，会让其他正在解码的请求多等几十到上百毫秒。")
print("用户视角就是'打字打到一半卡住了'。")

## 二、问题出在 Scheduler 的哪一行

回顾第 04 章的 `Scheduler.schedule()`：

```python
budget = self.max_num_batched_tokens
...
n = min(req.num_tokens_to_schedule(), budget)   # ← 就是这一行
```

一个长 prompt 进来时，`num_tokens_to_schedule()` 返回 2048。如果预算是 4096，它一口气全拿走，这一轮就变成一次超长前向。

**把预算调小到 256，同一行代码返回的就变成 256**——长 prompt 自动被切成多轮。没有第二个代码分支，没有开关。

下面用真实的 `EngineCore` 跑两遍，看同一行代码在不同预算下的行为。

In [ ]:
def run_episode(budget, long_len=2048, n_decode=8, prompt_len=32, dec_steps=24):
    """跑一段真实调度：8 条请求已进入稳态 decode，中途插入一条长 prompt。

    返回 (每轮迭代耗时列表, 长请求在第几轮拿到第一个 token, 长请求对象)。
    """
    g = torch.Generator().manual_seed(7)
    sched = Scheduler(max_num_seqs=32, max_num_batched_tokens=budget)
    engine = EngineCore(model, sched)

    for i in range(n_decode):
        prompt = torch.randint(0, model.cfg.vocab_size, (prompt_len,), generator=g).tolist()
        sched.add_request(Request(f"dec{i}", prompt, dec_steps))

    # 先跑 3 轮，让这 8 条进入稳态 decode
    for _ in range(3):
        engine.step()

    # 现在插入长请求
    long_prompt = torch.randint(0, model.cfg.vocab_size, (long_len,), generator=g).tolist()
    long_req = Request("long", long_prompt, 1)
    sched.add_request(long_req)

    iters, long_ttft_step = [], None
    while sched.has_unfinished():
        t0 = time.perf_counter()
        engine.step()
        sync()
        iters.append((time.perf_counter() - t0) * 1000)
        if long_ttft_step is None and long_req.output_token_ids:
            long_ttft_step = len(iters)
    return iters, long_ttft_step, long_req


def pct(xs, q):
    s = sorted(xs)
    return s[min(len(s) - 1, int(q * len(s)))]


results = {}
for budget in [4096, 256]:
    results[budget] = run_episode(budget)

print("8 条稳态 decode 请求 + 1 条 2048 token 长 prompt 插入\n")
print(f"{'指标':<30}{'预算 4096':>14}{'预算 256':>14}")
print("-" * 60)
for label, fn in [
    ("每轮耗时 p50 (ms)", lambda it, tt: pct(it, 0.50)),
    ("每轮耗时 p95 (ms)", lambda it, tt: pct(it, 0.95)),
    ("每轮耗时 最大 (ms)", lambda it, tt: max(it)),
    ("总轮数", lambda it, tt: len(it)),
    ("长请求第几轮拿到 token", lambda it, tt: tt),
]:
    a = fn(results[4096][0], results[4096][1])
    b = fn(results[256][0], results[256][1])
    if isinstance(a, float):
        print(f"{label:<30}{a:>14.1f}{b:>14.1f}")
    else:
        print(f"{label:<30}{a:>14}{b:>14}")

### 结果解读

你应该会看到：

| 现象 | 说明 |
|---|---|
| **每轮耗时最大值大幅下降** | 预算 4096 时有一轮要吞下整个 2048 token 的 prefill；预算 256 时每轮都均匀 |
| **长请求的 TTFT 变长** | 它要等好几轮才轮到算完，这是明确的代价 |
| **总轮数增加** | 切得更碎，轮数更多，但每轮更快 |

这就是 chunked prefill 的本质：**它不是优化，是重新分配**。用长请求自己的 TTFT，换所有其他请求的尾延迟。

值不值得看业务：长请求占比低（比如 5%）而 decode 请求海量时，这笔交易非常划算——5% 的用户多等一点，95% 的用户不再卡顿。

## 三、把调度轨迹打出来看

上面看的是结果，现在看**过程**——`Scheduler` 每轮到底排了多少 token。

In [ ]:
def trace_budget(budget, long_len=1024, steps=14):
    g = torch.Generator().manual_seed(11)
    sched = Scheduler(max_num_seqs=32, max_num_batched_tokens=budget)
    engine = EngineCore(model, sched)
    for i in range(4):
        prompt = torch.randint(0, model.cfg.vocab_size, (32,), generator=g).tolist()
        sched.add_request(Request(f"d{i}", prompt, 40))
    sched.add_request(Request("LONG", torch.randint(0, model.cfg.vocab_size, (long_len,),
                                                  generator=g).tolist(), 1))

    print(f"  {'step':>4}{'本轮总token':>14}{'LONG 本轮':>11}{'LONG 进度':>22}")
    for _ in range(steps):
        if not sched.has_unfinished():
            break
        out = sched.schedule()
        n_long = out.num_scheduled_tokens.get("LONG", 0)
        lr = next(r for r in sched.running if r.request_id == "LONG")
        progress = f"{lr.num_computed_tokens}/{lr.num_prompt_tokens}"
        print(f"  {sched.step_id:>4}{sum(out.num_scheduled_tokens.values()):>14}"
              f"{n_long:>11}{progress:>22}")
        engine.runner.execute_model(out)
        sched.update_from_output(out, {})      # 只关心调度，不关心采样结果


print("预算 = 1024（长 prompt 一次装得下）")
trace_budget(1024)
print()
print("预算 = 128（同样的长 prompt 被自动切成多轮）")
trace_budget(128)

两段轨迹唯一的区别就是预算数字，代码一行没改。

顺便注意 **`LONG 进度` 这一列**：它就是 `num_computed_tokens`，被切块之后它一格一格往前推，直到追上 prompt 长度才开始采样。这正是第 04 章说的——**读懂这个字段，chunked prefill 就不是一个独立机制了**。

## 四、什么时候不该用它

面试官喜欢追问边界，这三个都是真实的：

1. **chunk 切得太小**：每个 chunk 的矩阵乘太小，GPU 算力利用率骤降，总吞吐反而变差。切分粒度要在"阻塞时间"和"算力效率"之间取平衡。
2. **显存压力大时**：分块让更多请求同时处于"进行中"，KV cache 峰值占用上升，可能触发抢占。用显存换延迟，账要算清楚。
3. **本来就延迟不敏感**：离线批量推理只关心吞吐，chunked prefill 只带来额外调度开销。

还有一个容易忽略的：**它和 prefix caching 有重叠**。如果 prompt 大部分能命中缓存，prefill 本来就短（第 05 章讲过 `num_computed_tokens` 直接被推上去），chunk 的意义就不大。两个优化不要重复投入。

### 一个真实的调参顺序

因为这两个优化有重叠，线上的调优顺序应该是：

1. **先修 prompt 结构**，把 prefix cache 命中率拉起来——这是免费的。
2. **再看尾延迟是否还需要改善**，需要才开 chunked prefill。
3. 调 `max_num_batched_tokens` 时，同时盯 `num_requests_waiting` 和 TTFT p99，别把吞吐调崩了。

## 五、参数对照表

| 本章的东西 | vLLM 里的对应物 | 说明 |
|---|---|---|
| `Scheduler.max_num_batched_tokens` | `SchedulerConfig.max_num_batched_tokens` | 名字和语义完全一致 |
| `--enable-chunked-prefill` | 同名启动参数 | vLLM 里它是一个开关，开的本质是把预算调小 |
| `Request.num_computed_tokens` | 同名字段 | 分块进度就记录在这里 |
| `Scheduler.schedule()` 里的 `n = min(need, budget)` | `Scheduler` 里分配 token 预算的逻辑 | 就是这一行产生 chunked prefill |

**关键认知**：vLLM 里 `enable_chunked_prefill` 这个开关之所以存在，是因为开启后调度策略会变化（比如不允许一个请求独占整个 batch），但底层的切分机制就是 token 预算，没有第二种实现。

## 六、面试话术

**问：chunked prefill 为什么能改善尾延迟？代价是什么？**

- **问题**：prefill 是不可中断的一次大计算，长 prompt 独占 GPU，让同批正在解码的请求排队，表现为 TBT 尖刺。
- **机制（这里要答准）**：它不需要新机制。`Scheduler` 每轮有一个 token 预算 `max_num_batched_tokens`，长 prompt 装不下就被自然切成多轮，`num_computed_tokens` 记录进度，下一轮接着算。
- **收益**：把一次长阻塞摊成多次短阻塞，TBT p99 显著下降。
- **代价**：长请求自身 TTFT 变长；chunk 过小降低算力效率；同时进行中的请求变多，KV 显存峰值上升。
- **本质**：总计算量不变，是延迟在请求之间的**重新分配**。

最后那句"不是优化而是重新分配"，加上"它本质是 token 预算而不是新机制"，这两句加起来会让面试官确认你是真的读过源码，而不是看过几篇公众号。

**作业**

1. 把预算从 256 改成 64 和 1024，重跑 episode，找出你机器上"尾延迟"和"总算力效率"的平衡点。
2. 把 `n_decode` 从 8 改成 32，长 prefill 的阻塞效应是变强还是变弱？为什么？
3. 思考题：如果长请求的用户体验很重要（比如付费用户），设计什么机制既保住他的 TTFT 又保住其他人的尾延迟？（提示：vLLM 的 `priority` 参数 + 调度优先级 + 抢占）

**下一章**：换个方向提速——用一个小模型给大模型"打草稿"，也就是投机解码。